**Data Cleaning and Prearation**

*Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation
before analysis. This phase prepares the dataset for reliable analytics.*

In [1]:
import pandas as pd

*Converting Data fields to datetime format*

In [2]:
folder = r"C:\Users\khush\idx files"
reports_folder = r"C:\Users\khush\Desktop\IDX-Exchange\Reports"

sold = pd.read_csv(reports_folder + r"\sold_with_rates.csv", low_memory=False)
listings = pd.read_csv(reports_folder + r"\listings_with_rates.csv", low_memory=False)

In [3]:
sold['CloseDate'] = pd.to_datetime(sold['CloseDate'])
sold['PurchaseContractDate'] = pd.to_datetime(sold['PurchaseContractDate'])
sold['ListingContractDate'] = pd.to_datetime(sold['ListingContractDate'])
sold['ContractStatusChangeDate'] = pd.to_datetime(sold['ContractStatusChangeDate'])
listings['CloseDate'] = pd.to_datetime(listings['CloseDate'])
sold['PurchaseContractDate'] = pd.to_datetime(sold['PurchaseContractDate'])
sold['ListingContractDate'] = pd.to_datetime(sold['ListingContractDate'])
sold['ContractStatusChangeDate'] = pd.to_datetime(sold['ContractStatusChangeDate'])

In [4]:
sold[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object

In [5]:
listings[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   datetime64[us]
PurchaseContractDate                   str
ListingContractDate                    str
ContractStatusChangeDate               str
dtype: object

*Remove unnecessary or redundant columns*

In [6]:
cols_to_drop = [
    'TaxAnnualAmount',
    'AboveGradeFinishedArea', 
    'TaxYear',
    'ElementarySchoolDistrict',
    'CoveredSpaces',
    'BusinessType',
    'MiddleOrJuniorSchoolDistrict',
    'FireplacesTotal',
    'WaterfrontYN',
    'BelowGradeFinishedArea',
    'BasementYN',
    'LotSizeDimensions',
    'BuilderName',
    'BuildingAreaTotal',
    'CoBuyerAgentFirstName'
]

sold = sold.drop(columns=cols_to_drop, errors='ignore')
listings = listings.drop(columns=cols_to_drop, errors='ignore')

In [7]:
sold.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice',
       'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice',
       'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude',
       'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName',
       'BuyerOfficeAOR', 'YearBuilt', 'BuyerAgencyCompensationType',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BuyerAgencyCompensation', 'BedroomsTotal', 'ContractStatusChangeDate',
       'PurchaseContractDate', 'ListingContract

In [8]:
# Drop redundant columns from sold
sold = sold.drop(columns=[
    'LotSizeDimensions',
    'LotSizeArea',
    'LotSizeSquareFeet',
    'ListingKeyNumeric',
    'ListingId',
    'BuyerAgencyCompensationType',
    'BuyerAgencyCompensation',
    'OriginatingSystemName',
    'OriginatingSystemSubName'
], errors='ignore')

# Drop redundant columns from listings (including .1 duplicates)
listings = listings.drop(columns=[
    col for col in listings.columns if col.endswith('.1')
] + [
    'LotSizeDimensions',
    'LotSizeArea',
    'LotSizeSquareFeet',
    'ListingKeyNumeric',
    'ListingId',
    'BuyerAgencyCompensationType',
    'BuyerAgencyCompensation',
    'OriginatingSystemName',
    'OriginatingSystemSubName'
], errors='ignore')

print("Sold shape:", sold.shape)
print("Listings shape:", listings.shape)

Sold shape: (448253, 61)
Listings shape: (607724, 56)


*Remove or flag invalid numeric values:*

In [ ]:
sold["invalid_close_price"] = sold["ClosePrice"] <= 0
sold["invalid_living_area"] = sold["LivingArea"] <= 0
sold["invalid_days_on_market"] = sold["DaysOnMarket"] < 0
sold["invalid_bed_or_bath"] = (sold["BathroomsTotalInteger"] < 0) | (sold["BedroomsTotal"] < 0)
print("Invalid ClosePrice:", sold["invalid_close_price"].sum())
print("Invalid LivingArea:", sold["invalid_living_area"].sum())
print("Invalid DaysOnMarket:", sold["invalid_days_on_market"].sum())
print("Invalid Bed/Bath:", sold["invalid_bed_or_bath"].sum())

Invalid ClosePrice: 1
Invalid LivingArea: 165
Invalid DaysOnMarket: 51
Invalid Bed/Bath: 0


In [10]:
listings["invalid_close_price"] = listings["ClosePrice"] <= 0
listings["invalid_living_area"] = listings["LivingArea"] <= 0
listings["invalid_days_on_market"] = listings["DaysOnMarket"] < 0
listings["invalid_bed_or_bath"] = (listings["BathroomsTotalInteger"] < 0) | (listings["BedroomsTotal"] < 0)

print("Invalid ClosePrice:", listings["invalid_close_price"].sum())
print("Invalid LivingArea:", listings["invalid_living_area"].sum())
print("Invalid DaysOnMarket:", listings["invalid_days_on_market"].sum())
print("Invalid Bed/Bath:", listings["invalid_bed_or_bath"].sum())

Invalid ClosePrice: 0
Invalid LivingArea: 393
Invalid DaysOnMarket: 32
Invalid Bed/Bath: 0


*Consistency Checks*

In [11]:
sold["listing_after_close_flag"] = sold["ListingContractDate"] > sold["CloseDate"]
sold["purchase_after_close_flag"] = sold["PurchaseContractDate"] > sold["CloseDate"]
sold["negative_timeline_flag"] = sold["ListingContractDate"] > sold["PurchaseContractDate"]

In [12]:
print("Listing after close:", sold["listing_after_close_flag"].sum())
print("Purchase after close:", sold["purchase_after_close_flag"].sum())
print("Negative timeline:", sold["negative_timeline_flag"].sum())

Listing after close: 70
Purchase after close: 239
Negative timeline: 295


*Geographic data checks*

In [13]:
# Sold geographic flags
sold["missing_coordinates"] = (sold["Latitude"].isnull()) | (sold["Longitude"].isnull())
sold["zero_coordinates"] = (sold["Latitude"] == 0) | (sold["Longitude"] == 0)
sold["invalid_longitude"] = sold["Longitude"] > 0

print("Sold geographic flags:")
print("Missing coordinates:", sold["missing_coordinates"].sum())
print("Zero coordinates:", sold["zero_coordinates"].sum())
print("Invalid longitude:", sold["invalid_longitude"].sum())

# Listings geographic flags
listings["missing_coordinates"] = (listings["Latitude"].isnull()) | (listings["Longitude"].isnull())
listings["zero_coordinates"] = (listings["Latitude"] == 0) | (listings["Longitude"] == 0)
listings["invalid_longitude"] = listings["Longitude"] > 0

print("\nListings geographic flags:")
print("Missing coordinates:", listings["missing_coordinates"].sum())
print("Zero coordinates:", listings["zero_coordinates"].sum())
print("Invalid longitude:", listings["invalid_longitude"].sum())

Sold geographic flags:
Missing coordinates: 4388
Zero coordinates: 37
Invalid longitude: 31

Listings geographic flags:
Missing coordinates: 80824
Zero coordinates: 75
Invalid longitude: 92


In [14]:
sold["StateOrProvince"].value_counts()

StateOrProvince
CA    448223
AZ        10
OS         4
NV         2
CO         2
FL         2
ID         1
TX         1
GA         1
ME         1
NY         1
MO         1
TN         1
AR         1
VA         1
Name: count, dtype: int64

In [15]:
sold["out_of_state_flag"] = sold["StateOrProvince"] != "CA"
print("Out of state records:", sold["out_of_state_flag"].sum())

Out of state records: 30


In [16]:
sold.to_csv(reports_folder + r"\sold_cleaned.csv", index=False)
listings.to_csv(reports_folder + r"\listings_cleaned.csv", index=False)

In [ ]:
#future - look into flagged columns + investigate if other columns are valuable for our dashboard + look at lat + long columns

#look into listing + closing dates that are in the future

#some huge bathroom + bedroom numbers that need to be looked into 